# 01. Basic Network Drawing with NetworkX

This notebook introduces the **core NetworkX drawing functions** on a compact city network. The goal is to understand what each drawing function does, what its most important parameters control, and when one function is more useful than another.

**Learning goals**
- distinguish `nx.draw(...)` from `nx.draw_networkx(...)`
- understand why a `pos` dictionary matters
- move from one-shot drawing functions to layered node/edge rendering
- map node and edge attributes to size, color, and width
- use selective highlighting without overwhelming the graph

**Notebook flow**
1. load a compact warm-up network
2. compare the most common drawing entry points
3. reuse coordinates stored in the data
4. build a layered, data-driven figure
5. highlight only the few edges that deserve emphasis



In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.colors as mcolors
import networkx as nx
from networkx.readwrite import json_graph
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()
TEXT = apply_teaching_rc(grid=False, font_base=11.5)
FIG = make_figure_size_scale(
    focus=(7.2, 7.2),
    standard=(8.2, 5.2),
    wide=(11.4, 6.6),
    matrix=(11.2, 8.8),
)

NOTEBOOK_DIR = tutorials_dir / "04-networks"
DATA_DIR = NOTEBOOK_DIR / "data"


In [ ]:
# ── Visual defaults ────────────────────────────────────────────────────────
BASE_NODE_SIZE    = 42
BASE_NODE_FACE    = mcolors.to_hex(lighten_color(DV_PALETTE["blue"], 0.94))
BASE_NODE_EDGE    = "#355064"
BASE_NODE_LINEWIDTH = 0.9
BASE_EDGE_WIDTH   = 0.45

DEMO_NODE_SIZE    = 105
DEMO_NODE_FACE    = mcolors.to_hex(lighten_color(DV_PALETTE["blue"], 0.88))
DEMO_NODE_EDGE    = "#314252"
DEMO_EDGE_WIDTH   = 0.9

# Keyword bundle used by the early examples that share the same visual baseline.
common_params = dict(
    node_size=BASE_NODE_SIZE,
    width=BASE_EDGE_WIDTH,
    node_color=BASE_NODE_FACE,
    edgecolors=BASE_NODE_EDGE,
    linewidths=BASE_NODE_LINEWIDTH,
)

CITY_NODE_CMAP       = plt.cm.Blues
BASE_EDGE_COLOR      = "#C9D1DA"
HIGHLIGHT_EDGE_COLOR = "#223548"
CITY_NODE_EDGE       = DEMO_NODE_EDGE
CITY_LABEL_OFFSETS   = {
    "Los Angeles CA": (-26, -8),
    "Chicago IL":     (22, 24),
    "New York NY":    (35, 10),
}
LABEL_BBOX = dict(boxstyle="round,pad=0.22", fc="white", ec="#D5DDE7", lw=0.8, alpha=0.97)


# ── Helper functions ────────────────────────────────────────────────────────

def style_network_panel(ax, title=None, subtitle=None):
    """Place a title and optional subtitle above the axes and turn the frame off."""
    from textwrap import fill

    fig = ax.figure
    pos = ax.get_position()
    fig_width, fig_height = fig.get_size_inches()

    def pts_to_fig_y(points):
        return (points / 72.0) / fig_height

    def width_to_chars(width_fraction, chars_per_inch=11):
        panel_width_in = max(fig_width * width_fraction, 2.0)
        return max(28, int(panel_width_in * chars_per_inch))

    wrapped_title    = fill(title,    width=width_to_chars(pos.width, chars_per_inch=9)) if title    else None
    wrapped_subtitle = fill(subtitle, width=width_to_chars(pos.width))                   if subtitle else None

    title_lines    = wrapped_title.count("\n")    + 1 if wrapped_title    else 0
    subtitle_lines = wrapped_subtitle.count("\n") + 1 if wrapped_subtitle else 0

    title_line_pts    = TEXT["panel_title"] * 1.08
    subtitle_line_pts = TEXT["annotation"]  * 1.22
    top_margin_pts    = 2
    gap_pts           = 3 if (wrapped_title and wrapped_subtitle) else 0
    bottom_margin_pts = 4 if (wrapped_title or wrapped_subtitle)  else 0

    header_pts = top_margin_pts + bottom_margin_pts
    if wrapped_title:    header_pts += title_lines    * title_line_pts
    if wrapped_subtitle: header_pts += gap_pts        + subtitle_lines * subtitle_line_pts

    header_height = min(
        pts_to_fig_y(max(header_pts, 18 if (wrapped_title or wrapped_subtitle) else 0)),
        pos.height * 0.18,
    )
    new_height = max(pos.height - header_height, pos.height * 0.68)
    ax.set_position([pos.x0, pos.y0, pos.width, new_height])

    current_top = pos.y0 + pos.height - pts_to_fig_y(top_margin_pts)
    if wrapped_title:
        fig.text(pos.x0, current_top, wrapped_title,
                 ha="left", va="top", fontsize=TEXT["panel_title"], color="#17212B")
        current_top -= pts_to_fig_y(title_lines * title_line_pts + gap_pts)
    if wrapped_subtitle:
        fig.text(pos.x0, current_top, wrapped_subtitle,
                 ha="left", va="top", fontsize=TEXT["annotation"], color=DV_PALETTE["gray"])

    ax.set_axis_off()
    return ax


def scale(values, minv=50, maxv=200):
    """Min-max rescale an array of values to the interval [minv, maxv]."""
    values = np.asarray(list(values), dtype=float)
    if np.allclose(values, values[0]):
        return np.full(values.shape, (minv + maxv) / 2)
    lo, hi = values.min(), values.max()
    return minv + (maxv - minv) * (values - lo) / (hi - lo)


def annotate_named_places(ax, pos, labels, offsets):
    """Annotate selected nodes with callout labels at fixed pixel offsets."""
    for node, label in labels.items():
        dx, dy = offsets[node]
        ax.annotate(
            label,
            xy=pos[node],
            xytext=(dx, dy),
            textcoords="offset points",
            ha="left" if dx >= 0 else "right",
            va="bottom" if dy >= 0 else "top",
            fontsize=TEXT["direct_label"],
            fontweight="bold",
            color="#111111",
            bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.96),
            arrowprops=dict(arrowstyle="-", color="#6B7280", lw=1.0,
                            shrinkA=2, shrinkB=3, alpha=0.8),
            annotation_clip=False,
        )
    return ax


## Loading a Compact Warm-Up Network

The examples use a network of major US cities. Nodes carry `location` (longitude, latitude) and `population` attributes; edges carry a `weight`. That combination lets us exercise both topological and spatial views of the same graph.

In [ ]:
with (DATA_DIR / "major_us_cities.json").open() as fh:
    G = json_graph.node_link_graph(json.load(fh), edges="links")

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## Getting Started: Drawing a Graph

A first network drawing is usually exploratory. At this stage the goal is not a final figure, but a quick answer to simple questions:
- Did the graph load correctly?
- Is the network dense or sparse?
- Are there obvious clusters or hubs?

Quick drawing functions are appropriate for this stage because speed matters more than polish.


### Function Guide: `nx.draw`

`nx.draw(G, pos=None, ax=None, **kwds)` is the minimal entry point: one call, one figure, no separate layers. It is best for a *first structural sketch*, not for a polished figure.

**Most useful parameters**
- `pos` — node position dictionary; omit to let NetworkX compute a spring layout
- `ax` — target Matplotlib axes
- `node_size`, `node_color`, `edgecolors`, `linewidths` — node appearance
- `width`, `edge_color` — edge appearance
- `with_labels` — draw node labels (default `True`; pass `False` to suppress)

Any extra keyword argument is forwarded to the underlying drawing machinery. That makes `nx.draw` convenient but also a little opaque — once the figure needs separate control over nodes, edges, and labels, move on to the explicit layer functions.


### Example 1: Minimal `nx.draw`

The first example is intentionally minimal. It shows what `nx.draw` gives you with almost no decisions from you beyond choosing the axes.

In [ ]:
# Example 1: the smallest possible structural sketch
fig, ax = plt.subplots(figsize=FIG["standard"])
nx.draw(G, ax=ax)
style_network_panel(
    ax,
    "Minimal `nx.draw` sketch",
    "Useful for a first inspection, but too implicit for serious design work.",
)

### Example 2: `nx.draw` with explicit positions and a few style parameters

The next example is still `nx.draw`, but now the main parameters are explicit.

Read it as a checklist:
- `pos` fixes the node geometry
- `node_size` changes marker area
- `node_color` and `edge_color` control the visual palette
- `width` controls edge thickness
- `with_labels=False` keeps the first sketch quiet

In [ ]:
quick_layout_pos = nx.spring_layout(G, seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=FIG["standard"])
nx.draw(
    G,
    pos=quick_layout_pos,
    ax=ax,
    node_size=DEMO_NODE_SIZE,
    node_color=DEMO_NODE_FACE,
    edgecolors=DEMO_NODE_EDGE,
    linewidths=1.0,
    edge_color=BASE_EDGE_COLOR,
    width=DEMO_EDGE_WIDTH,
    with_labels=False,
)
style_network_panel(
    ax,
    "`nx.draw` with explicit layout and styling",
    "Change `DEMO_NODE_SIZE` and `DEMO_NODE_FACE` above to restyle this figure without rewriting the plotting call.",
)

### Function Guide: `nx.draw_networkx`

`nx.draw_networkx(G, pos=None, arrows=None, with_labels=True, **kwds)` is the richer all-in-one helper. It exposes labeling controls directly and accepts node/edge sublists, making it easier to tune a diagnostic figure without splitting the drawing into explicit layers.

**Additional parameters over `nx.draw`**
- `nodelist`, `edgelist` — draw only a subset of nodes or edges
- `font_size`, `font_color`, `font_weight`, `font_family` — label typography
- `bbox` — label text-box styling
- `arrows`, `arrowstyle`, `arrowsize` — directed-graph arrows

**When to use it vs. the separate draw functions**
Use `nx.draw_networkx` when the graph is small or when node identity matters at a glance. Switch to `draw_networkx_nodes` / `_edges` / `_labels` once nodes, edges, and labels need different visual roles, or when you want emphasized layers on top of a quiet background network.


### Example 3: `nx.draw_networkx` as a richer all-in-one call

This example uses the same graph and the same abstract layout as before, but now the function call exposes more of the common controls directly.

Notice what changes:
- labels appear because `with_labels=True`
- `font_size` and `font_weight` matter immediately
- edge and node styling can still be set in one place

This is stronger than the bare `nx.draw` call, but it still mixes several visual roles into one function.

In [ ]:
fig, ax = plt.subplots(figsize=FIG["standard"])
nx.draw_networkx(
    G,
    pos=quick_layout_pos,
    ax=ax,
    with_labels=True,
    node_size=DEMO_NODE_SIZE + 15,
    node_color=DEMO_NODE_FACE,
    edgecolors=DEMO_NODE_EDGE,
    linewidths=1.0,
    edge_color=BASE_EDGE_COLOR,
    width=DEMO_EDGE_WIDTH,
    font_size=8,
    font_weight="semibold",
)
style_network_panel(
    ax,
    "`nx.draw_networkx` with labels",
    "The same node styling can be reused here, while labels add a second visual layer.",
)

## Using Real-World Coordinates

When nodes represent places, we often have two competing but legitimate views of the same network:
- an **abstract** view, where layout reveals relational structure
- a **spatial** view, where layout preserves geography

The key question is: what do you want the reader to learn from the figure? If the answer is about route geography, regional isolation, or spatial reach, geographic coordinates are usually the stronger choice.


In [ ]:
city_geo_pos = nx.get_node_attributes(G, "location")

### `pos` from Real Coordinates

When nodes represent places with known coordinates, pass a `{node: (x, y)}` dictionary as `pos` instead of calling a layout algorithm. Here, `nx.get_node_attributes(G, "location")` returns exactly that dictionary; the values are `(longitude, latitude)` pairs that map directly to plot axes.

The next two figures also use `common_params`, a keyword-argument dictionary defined at the top of the notebook. It bundles the shared node and edge defaults into one reusable object so the same visual baseline can be applied across multiple figures without repeating every styling keyword:

```python
common_params = dict(
    node_size=BASE_NODE_SIZE,
    width=BASE_EDGE_WIDTH,
    node_color=BASE_NODE_FACE,
    edgecolors=BASE_NODE_EDGE,
    linewidths=BASE_NODE_LINEWIDTH,
)
```

When the only difference between two figures is *what information they convey*, not how they are styled, a shared `**common_params` unpacking keeps the plotting code clean and the visual baseline consistent.


In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw_networkx(G, city_geo_pos, ax=ax, **common_params)
style_network_panel(ax, "City network in geographic coordinates")


### Geographic layout: labeled vs. unlabeled

Two views of the same graph reveal different things. The labeled view above helps orient the reader but introduces visual clutter as the network grows. The unlabeled view below exposes structural patterns — which cities are central hubs, where the network is sparse — without the weight of text competing for attention.

The choice between them depends on the question the figure is answering: use labels when **node identity** is the message; drop them when **topology or spatial density** is the focus.


In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw(G, city_geo_pos, ax=ax, **common_params)
style_network_panel(ax, "Same coordinates, fewer annotations")


## Making Visualizations Data-Driven

The network becomes analytically richer once visual variables are tied to measured attributes. This is one of the core best practices in network visualization:

- map ordered data to ordered visual channels
- avoid using the same visual channel for too many different messages
- make the legend or explanation clear enough that a reader can recover the encoding

In the next cell we apply three encodings at once, but each one answers a different question:
- node size asks which cities are large
- node color asks which cities are structurally connected
- edge width asks which ties are strong


### Drawing Nodes, Edges, and Labels Separately

Once a network carries real information, all-in-one functions become limiting. The pattern is to call each drawing function independently and layer the results on the same axes. The call order sets the z-order: later calls appear on top.

**`nx.draw_networkx_nodes(G, pos, nodelist=None, node_size=300, node_color='#1f78b4', node_shape='o', alpha=None, linewidths=1.0, edgecolors=None, ax=None)`**\
Draws only the nodes. Pass a numeric array to `node_color` together with `cmap` for a colormap mapping, or a list of hex strings for per-node colors. Use `nodelist` to highlight a subset by calling this function a second time.

**`nx.draw_networkx_edges(G, pos, edgelist=None, width=1.0, edge_color='k', style='solid', alpha=None, arrows=None, arrowstyle='-|>', arrowsize=10, node_size=300, connectionstyle=None, ax=None)`**\
Draws only the edges. Call it twice to separate quiet background context from emphasized corridors. The `node_size` parameter matters here because endpoint clipping is computed relative to node markers — pass the same value you used in `draw_networkx_nodes`.

**`nx.draw_networkx_labels(G, pos, labels=None, font_size=12, font_color='k', font_weight='normal', bbox=None, horizontalalignment='center', verticalalignment='center', ax=None)`**\
Draws only labels. Pass a `labels` dictionary to annotate only a selected subset of nodes rather than all of them.

**Why split the layers?**
- edges as quiet context first, then nodes as the main information layer
- emphasized edges or nodes are extra calls on the same axes, not rewrites of the whole plot
- labels are added only where they help — density and overlap are under your control


### Example 4: `draw_networkx_nodes` in isolation

This example keeps edges extremely light and uses `draw_networkx_nodes` as the star of the figure. The purpose is to make the node parameters easy to read:
- a numeric color mapping
- a size mapping
- explicit border styling

In [ ]:
# Attributes used across Examples 4–6 and both layered figures below.
# Define them once here so the plotting cells stay free of data-preparation code.
degree_values  = pd.Series(dict(G.degree()))
population     = pd.Series(nx.get_node_attributes(G, "population"))
edge_weights   = pd.Series(nx.get_edge_attributes(G, "weight"))

key_city_labels = {
    "Los Angeles CA": "Los Angeles",
    "Chicago IL":     "Chicago",
    "New York NY":    "New York",
}
key_city_nodes = list(key_city_labels)

# Normalize degree for a sequential color scale.
norm = mcolors.Normalize(vmin=degree_values.min(), vmax=degree_values.max())

# Rescale raw attributes to practical plotting ranges.
city_node_size     = scale(population.values, minv=60, maxv=760)
city_node_size_map = dict(zip(G.nodes(), city_node_size))
edge_width         = scale(edge_weights.values, minv=0.4, maxv=3.4)

# Subsets used by Example 5 (top-weight edges) and Example 6 (key city labels).
edge_demo_subset  = edge_weights.sort_values(ascending=False).head(6).index.tolist()
label_demo_subset = {node: key_city_labels[node] for node in key_city_nodes}

In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    edge_color=BASE_EDGE_COLOR,
    width=0.6,
    alpha=0.18,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    node_size=city_node_size,
    node_color=degree_values.values,
    cmap=CITY_NODE_CMAP,
    edgecolors=CITY_NODE_EDGE,
    linewidths=1.0,
    alpha=0.96,
    ax=ax,
)
sm = plt.cm.ScalarMappable(cmap=CITY_NODE_CMAP, norm=norm)
plt.colorbar(sm, shrink=0.70, ax=ax, pad=0.02, label="Node degree")
ax.set_aspect("equal")
style_network_panel(
    ax,
    "`draw_networkx_nodes`: nodes as the main information layer",
    "Node size encodes population and node color encodes degree, while edges stay deliberately quiet.",
)


### Example 5: `draw_networkx_edges` in isolation

Now the logic flips: nodes become quiet anchors and `draw_networkx_edges` does the interpretive work.

This is the right function when the question is about:
- which ties are strongest
- which corridors deserve emphasis
- whether some edges should be styled differently from the background network

In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    width=edge_width,
    edge_color=BASE_EDGE_COLOR,
    alpha=0.25,
    ax=ax,
)
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    edgelist=edge_demo_subset,
    width=4.2,
    edge_color=HIGHLIGHT_EDGE_COLOR,
    style="solid",
    alpha=0.70,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    node_size=110,
    node_color="white",
    edgecolors=CITY_NODE_EDGE,
    linewidths=1.0,
    ax=ax,
)
ax.set_aspect("equal")
style_network_panel(
    ax,
    "`draw_networkx_edges`: emphasizing only a selected edge subset",
    "The strongest corridors are drawn separately from the background network so the highlighting rule stays explicit.",
)

### Example 6: `draw_networkx_labels` in isolation

Labeling is strongest when it is explicit and selective. The label function is therefore best understood as its own layer, not as a side effect of a one-call graph drawing.

This example shows three key label ideas from the official API:
- pass a custom `labels` dictionary
- style the text with `font_size`, `font_weight`, and `bbox`
- label only the few nodes that genuinely help the reader

In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    width=0.8,
    edge_color=BASE_EDGE_COLOR,
    alpha=0.22,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    node_size=95,
    node_color="white",
    edgecolors=CITY_NODE_EDGE,
    linewidths=1.0,
    ax=ax,
)
nx.draw_networkx_labels(
    G,
    city_geo_pos,
    labels=label_demo_subset,
    font_size=11,
    font_weight="semibold",
    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="#D5DDE7", lw=0.8),
    horizontalalignment="left",
    verticalalignment="bottom",
    ax=ax,
)
ax.set_aspect("equal")
style_network_panel(
    ax,
    "`draw_networkx_labels`: labels as a separate, selective layer",
    "A custom label dictionary keeps annotation focused on the few nodes worth naming.",
)

### Step-by-Step Reading of the First Layered Figure

The next plot uses the layered drawing pattern for the first time:
- edges go down first as light structure
- nodes carry the main quantitative encodings
- a tiny highlighted subset receives callouts
- the colorbar explains the ordered color mapping


In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    width=edge_width,
    edge_color=BASE_EDGE_COLOR,
    alpha=0.42,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    node_size=city_node_size,
    node_color=degree_values.values,
    alpha=0.94,
    edgecolors=CITY_NODE_EDGE,
    linewidths=1.0,
    cmap=CITY_NODE_CMAP,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    nodelist=key_city_nodes,
    node_size=[city_node_size_map[node] * 1.58 for node in key_city_nodes],
    node_color="none",
    edgecolors=DV_PALETTE["orange"],
    linewidths=2.2,
    ax=ax,
)
annotate_named_places(ax, city_geo_pos, key_city_labels, CITY_LABEL_OFFSETS)

sm = plt.cm.ScalarMappable(cmap=CITY_NODE_CMAP, norm=norm)
plt.colorbar(sm, shrink=0.60, ax=ax, pad=0.06, label="Node degree")

style_network_panel(
    ax,
    "Population, degree, and edge weight in one view",
    "Population sets node size, degree sets fill color, and tie strength sets edge width.",
)


### Highlighting and Annotating Networks

Highlighting works only when it is visually restrained. The goal is not to make important edges scream louder than everything else. The goal is to create a clean foreground-background separation.

In the next figure we use a more disciplined pattern:
- keep the whole network as light context
- accent only the strongest corridors
- keep labels outside the densest part of the graph with short leader lines


### Isolate the Strongest Corridors, Then Draw

Two steps: first compute the highlight subset from edge weights (here, the top 5% by weight), then draw the background network and the highlighted edges as separate layers. Keeping the selection rule out of the plotting call makes it trivial to change the threshold later without touching the figure code.

In [ ]:
top_edges = edge_weights[edge_weights >= edge_weights.quantile(0.95)].index.tolist()

In [ ]:
fig, ax = plt.subplots(figsize=FIG["wide"])

nx.draw_networkx_edges(
    G,
    city_geo_pos,
    width=edge_width,
    edge_color=BASE_EDGE_COLOR,
    alpha=0.28,
    ax=ax,
)
nx.draw_networkx_edges(
    G,
    city_geo_pos,
    edgelist=top_edges,
    edge_color=HIGHLIGHT_EDGE_COLOR,
    alpha=0.58,
    width=4.0,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    node_size=city_node_size,
    node_color=degree_values.values,
    alpha=0.94,
    edgecolors=CITY_NODE_EDGE,
    linewidths=1.0,
    cmap=CITY_NODE_CMAP,
    ax=ax,
)
nx.draw_networkx_nodes(
    G,
    city_geo_pos,
    nodelist=key_city_nodes,
    node_size=[city_node_size_map[node] * 1.58 for node in key_city_nodes],
    node_color="none",
    edgecolors=DV_PALETTE["orange"],
    linewidths=2.2,
    ax=ax,
)
annotate_named_places(ax, city_geo_pos, key_city_labels, CITY_LABEL_OFFSETS)

sm = plt.cm.ScalarMappable(cmap=CITY_NODE_CMAP, norm=norm)
plt.colorbar(sm, shrink=0.70, ax=ax, pad=0.06, label="Node degree")
ax.set_aspect("equal")
style_network_panel(
    ax,
    "Selective highlighting beats all-at-once labeling",
    "A small number of emphasized corridors is enough to orient the reader.",
)


## Takeaways

- `nx.draw` is the quickest structural sketch, useful for first inspection.
- `nx.draw_networkx` is a richer all-in-one diagnostic function, especially for small labeled graphs.
- `draw_networkx_nodes`, `draw_networkx_edges`, and `draw_networkx_labels` are the core functions to learn if you want real control.
- `pos` is the central object in the drawing workflow: it separates geometry from styling.
- Strong network figures are built layer by layer, with each function doing one clear job.

Official references used in this notebook:
- [draw](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.nx_pylab.draw.html)
- [draw_networkx](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.nx_pylab.draw_networkx.html)
- [draw_networkx_nodes](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.nx_pylab.draw_networkx_nodes.html)
- [draw_networkx_edges](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.nx_pylab.draw_networkx_edges.html)
- [draw_networkx_labels](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.nx_pylab.draw_networkx_labels.html)

